# Avian Interaction Extraction - Structured Output

In [1]:
import os 
import requests
import json
from IPython.display import Markdown, display
import pandas as pd

from src.util import show_table, show_response_stats

OLLAMA_HOST = "127.0.0.1:40063"
os.environ["OLLAMA_HOST"] = "127.0.0.1:40063"
print(OLLAMA_HOST)

LLM_MODEL = "llama3.2:3b"


127.0.0.1:40063


## Making a basic chat handler 
We define a simple function that will allow us to send chat-style prompts to the local LLM server and returns the full response JSON, optionally including system context and example chat pairs (one-shot or few-shot prompting). 

The `/api/generate` endpoint is used primarily for one-off tasks, or single self-contained questions/queries. The `/api/chat` endpoint is primarily used for back-and-forth chatting, with the ability to preserve and manage chat history through the `messages` array. 

> Ollama’s /api/chat endpoint is stateless by default, and must be managed through `messages`

Here are some of the response items, with their definitions. See [Ollama /api/chat](https://docs.ollama.com/api/chat) for more information. 

`prompt_eval_count`: Number of tokens in the prompt

`eval_count`: Number of tokens generated in the response

`total_duration`: Total time spent generating in nanoseconds

`prompt_eval_duration`: Time spent evaluating the prompt in nanoseconds

`eval_duration`: Time spent generating tokens in nanoseconds

In [29]:
def call_llm(user_prompt, system_prompt = None, example_pair = None):
    """
    Send prompt to llm and return response
    """

    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    if example_pair:
        example_input, example_output = example_pair
        messages.append({"role": "user", "content": example_input})
        messages.append({"role": "assistant", "content": example_output})

    messages.append({"role": "user", "content": user_prompt})
        
    try:
        response = requests.post(
            f"http://{OLLAMA_HOST}/api/chat",
            json={
                "model": LLM_MODEL,
                "stream": False,
                "options": {
                    "num_ctx": 32768
                },
                "messages": messages
            }
        )
        data = response.json()
        return data #response.json()['message']['content']
        
    except Exception as e:
        print(f"Error during llm call: {e}")
        return None

In this example below, you could use curl in the terminal to get a comprable response (here using generate endpoint)
```
curl http://localhost:40063/api/generate -d '{
  "model": "llama3.2:3b",
  "prompt": "Why is the sky blue?",
  "stream": false,
  "options": {
    "num_ctx": 2048
  }
}'
```

In [16]:
response = call_llm("Why is the sky blue?")
print(response)

{'model': 'llama3.2:3b', 'created_at': '2026-03-27T17:56:47.218612945Z', 'message': {'role': 'assistant', 'content': "The sky appears blue to our eyes because of a phenomenon called Rayleigh scattering. Here's a simplified explanation:\n\n1. **Sunlight**: When sunlight enters Earth's atmosphere, it consists of a spectrum of colors, including all the colors of the visible light spectrum (red, orange, yellow, green, blue, indigo, and violet).\n2. **Scattering**: As sunlight travels through the atmosphere, it encounters tiny molecules of gases such as nitrogen (N2) and oxygen (O2). These molecules are much smaller than the wavelength of light.\n3. **Rayleigh scattering**: The smaller molecules scatter the shorter (blue) wavelengths of light more than the longer (red) wavelengths. This is known as Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century.\n4. **Blue light dominates**: As a result of this scattering, the blue light

You can use the `show_response_stats` helper function to display some basic stats (e.g., duration and token counts) in a table. 

In [17]:
show_response_stats(response)


| Metric | Description | Units | Value |
|--------|-------------|-------|-------|
| `prompt_eval_count` | Number of tokens in the prompt | tokens | 31 |
| `eval_count` | Number of tokens generated in the response | tokens | 416 |
| `total_duration` | Total time spent for the request | nanoseconds | 13038284674 |
| `prompt_eval_duration` | Time spent evaluating the prompt | nanoseconds | 2034194856 |
| `eval_duration` | Time spent generating tokens | nanoseconds | 1906333108 |


You may add system context to the prompt through the `system_prompt` input. 

In [5]:
system_prompt = "You are an expert atmospheric physicist. Give concise answers, limiting your responses to 2-3 sentences." 
response = call_llm("Why is the sky blue?", system_prompt=system_prompt)
print(response)

{'model': 'llama3.2:3b', 'created_at': '2026-03-18T08:12:55.492739071Z', 'message': {'role': 'assistant', 'content': "The sky appears blue due to a phenomenon called Rayleigh scattering, where shorter (blue) wavelengths of light are scattered more efficiently by the tiny molecules of gases in the Earth's atmosphere. This scattering effect gives the sky its characteristic blue color during the daytime. The amount of scattering decreases with longer wavelengths, resulting in the red and orange hues seen during sunrise and sunset."}, 'done': True, 'done_reason': 'stop', 'total_duration': 1553105912, 'load_duration': 223634943, 'prompt_eval_count': 52, 'prompt_eval_duration': 45133177, 'eval_count': 74, 'eval_duration': 1229170636}


## Interaction Extraction
We now will work on extracting information in a structured format. The use-case here is extracting species interactions from avian interaction text excerpts. First, let's look at an example input-output pair we'll use.

In [25]:
#load system prompt and user-assistant example pair
with open("./data/system.txt", "r") as f:
    system_prompt = f.read().strip()
    
with open("./data/example_input.txt", "r") as f:
    example_input = f.read().strip()

with open("./data/example_output.txt", "r") as f:
    example_output = f.read().strip()


print("Example input: ", example_input)
print("\n\nExample output:\n" + example_output)

Example input:  Actor species (species 1) common name: Cooper's Hawk, scientific name: Astur cooperii, genus combo name: Astur cooperii. Major Food Items: See Table 2. Relative importance of birds versus mammals varies across regions and among habitat types. Almost all data from breeding season, mostly from southeastern, midwestern and southwestern United States, and southwestern British Columbia. Of 12 studies across North America, 10 (83%) reported bird prey to be dominant, ranging from 56% to 96% of total items tallied (see summary in 1). The most common from smaller to larger: House Sparrow (example: ML81132121) (especially in cities such as Victoria, British Columbia; Milwaukee, Wisconsin; Grand Forks, North Dakota; and Albuquerque, New Mexico; 50, RNR); the Northern Cardinal (Cardinalis cardinalis) and the Northern Mockingbird (Mimus polyglottos); especially in rural Florida; 46); and the American Robin (example: ML40321891), European Starling (example: ML51466291), Blue Jay (Cya

In [26]:
show_table(example_output)

| species1_common | species1_scientific | genus_species_combo_sp1 | species2_common | species2_scientific | genus_species_combo_sp2 | label(type_of_interaction) | effect_on_sp1 | effect_on_sp2 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Cooper's Hawk | Astur cooperii | Astur cooperii | (unid. Red / Yellow Shafted) Northern Flicker | Colaptes auratus | Colaptes auratus auratus | predation | 1 | -1 |
| Cooper's Hawk | Astur cooperii | Astur cooperii | Eurasian Collared-Dove | Streptopelia decaocto | Streptopelia decaocto | predation | 1 | -1 |

Now let's use the example input for our user input, to see the extracted interaction table. Note in this specific response, it returned a nearly identical table as the example ouput. However, this will vary with each run, so you may see different results (e.g., more or less interactions). 

In [30]:
response = call_llm(example_input, system_prompt=system_prompt, example_pair=(example_input, example_output))

In [31]:
out = response['message']['content']
print(out)

species1_common|species1_scientific|genus_species_combo_sp1|species2_common|species2_scientific|genus_species_combo_sp2|label(type_of_interaction)|effect_on_sp1|effect_on_sp2
Cooper's Hawk|Astur cooperii|Astur cooperii|(unid. Red / Yellow Shafted) Northern Flicker|Colaptes auratus|Colaptes auratus auratus|predation|1|-1
Cooper's Hawk|Astur cooperii|Astur cooperii|Eurasian Collared-Dove|Streptopelia decaocto|Streptopelia decaocto|predation|1|-1


In [32]:
show_table(out)

| species1_common | species1_scientific | genus_species_combo_sp1 | species2_common | species2_scientific | genus_species_combo_sp2 | label(type_of_interaction) | effect_on_sp1 | effect_on_sp2 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Cooper's Hawk | Astur cooperii | Astur cooperii | (unid. Red / Yellow Shafted) Northern Flicker | Colaptes auratus | Colaptes auratus auratus | predation | 1 | -1 |
| Cooper's Hawk | Astur cooperii | Astur cooperii | Eurasian Collared-Dove | Streptopelia decaocto | Streptopelia decaocto | predation | 1 | -1 |

We'll now look at a single text excerpt that is not included in the example pair.

In [33]:
with open(f"./data/avian_interaction_docs/para_{1}.txt", "r") as f:
    user_prompt = f.read().strip()
print(user_prompt)

Actor species (species 1) common name: American Goshawk, scientific name: Astur atricapillus, genus combo name: Astur atricapillus. Major Food Items: Southern populations may depend less on mammalian prey than northern populations, but there are exceptions (Table 2; Boal and Mannan 1994). In boreal regions, grouse are fundamental foods, accounting for up to 80% of prey, particularly for chicks and juveniles. Although potential prey species are extensive (Appendix 1), a few taxa are particularly prevalent in most diets. Dominant mammalian prey in North America include tree squirrels (Douglas squirrel [Tamiasciurus douglasii], red squirrel [T. hudsonicus], gray squirrel [Sciurus carolinensis], tassel-eared squirrel [S. aberti], northern flying squirrel [Glaucomys sabrinus]); ground squirrels (golden-mantled ground squirrel [Spermophilus lateralis], Richardson's ground squirrel [S. richardsonii], Belding's ground squirrel [S. beldingi], Townsend's ground squirrel [S. townsendii]); and lag

In [34]:
response = call_llm(user_prompt, system_prompt=system_prompt, example_pair=(example_input, example_output))
out = response['message']['content']
print(out)

species1_common|species1_scientific|genus_species_combo_sp1|species2_common|species2_scientific|genus_species_combo_sp2|label(type_of_interaction)|effect_on_sp1|effect_on_sp2
American Goshawk|Astur atricapillus|Astur atricapillus|(unid. Red / Yellow Shafted) Northern Flicker|Colaptes auratus|Colaptes auratus auratus|predation|1|-1
American Goshawk|Astur atricapillus|Astur atricapillus|Eurasian Collared-Dove|Streptopelia decaocto|Streptopelia decaocto|predation|1|-1


In [35]:
show_table(out)

| species1_common | species1_scientific | genus_species_combo_sp1 | species2_common | species2_scientific | genus_species_combo_sp2 | label(type_of_interaction) | effect_on_sp1 | effect_on_sp2 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| American Goshawk | Astur atricapillus | Astur atricapillus | (unid. Red / Yellow Shafted) Northern Flicker | Colaptes auratus | Colaptes auratus auratus | predation | 1 | -1 |
| American Goshawk | Astur atricapillus | Astur atricapillus | Eurasian Collared-Dove | Streptopelia decaocto | Streptopelia decaocto | predation | 1 | -1 |

### Adding structured output format
Instead of specifying structured output / format in the system prompt, we can add the format as a JSON schema in the `/api/chat` requests. Here we re-define `call-llm` to use the desired format for the extracted interactions table.

In [36]:
def call_llm(user_prompt, system_prompt = None, example_pair = None):
    """
    Send prompt to llm and return response
    """

    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    if example_pair:
        example_input, example_output = example_pair
        messages.append({"role": "user", "content": example_input})
        messages.append({"role": "assistant", "content": example_output})

    messages.append({"role": "user", "content": user_prompt})

    # Define the structured output schema for species interactions
    output_format = {
        "type": "object",
        "properties": {
            "interactions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "species1_common": {"type": "string"},
                        "species1_scientific": {"type": "string"},
                        "genus_species_combo_sp1": {"type": "string"},
                        "species2_common": {"type": "string"},
                        "species2_scientific": {"type": "string"},
                        "genus_species_combo_sp2": {"type": "string"},
                        "label": {"type": "string"},
                        "effect_on_sp1": {"type": "integer"},
                        "effect_on_sp2": {"type": "integer"}
                    },
                    "required": [
                        "species1_common",
                        "species1_scientific",
                        "genus_species_combo_sp1",
                        "species2_common",
                        "species2_scientific",
                        "genus_species_combo_sp2",
                        "label",
                        "effect_on_sp1",
                        "effect_on_sp2"
                    ]
                }
            }
        },
        "required": ["interactions"]
    }
    
    try:
        response = requests.post(
            f"http://{OLLAMA_HOST}/api/chat",
            json={
                "model": LLM_MODEL,
                "stream": False,
                "options": {
                    "num_ctx": 32768
                },
                "messages": messages,
                "format": output_format  # add structured output
            }
        )
        data = response.json()
        return data # response.json()['message']['content']
        
    except Exception as e:
        print(f"Error during llm call: {e}")
        return None

In [37]:
with open("./data/system-nofmt.txt", "r") as f:
    system_prompt_nofmt = f.read().strip()

response = call_llm(user_prompt, system_prompt=system_prompt_nofmt, example_pair=(example_input, example_output))
out = response['message']['content']
out_json = json.loads(out)
print(out_json)

{'interactions': [{'species1_common': 'American Goshawk', 'species1_scientific': 'Astur atricapillus', 'genus_species_combo_sp1': 'Astur atricapillus', 'species2_common': 'Northern Flicker', 'species2_scientific': 'Colaptes auratus', 'genus_species_combo_sp2': 'Colaptes auratus', 'label': 'predation', 'effect_on_sp1': -1, 'effect_on_sp2': 1}, {'species1_common': 'American Goshawk', 'species1_scientific': 'Astur atricapillus', 'genus_species_combo_sp1': 'Astur atricapillus', 'species2_common': 'Blue Jay', 'species2_scientific': 'Cyanocitta cristata', 'genus_species_combo_sp2': 'Cyanocitta cristata', 'label': 'predation', 'effect_on_sp1': -1, 'effect_on_sp2': 1}, {'species1_common': 'American Goshawk', 'species1_scientific': 'Astur atricapillus', 'genus_species_combo_sp1': 'Astur atricapillus', 'species2_common': "Steller's Jay", 'species2_scientific': 'Cyanocorax stelleri', 'genus_species_combo_sp2': 'Cyanocorax stelleri', 'label': 'predation', 'effect_on_sp1': -1, 'effect_on_sp2': 1}, 

In [38]:
df = pd.DataFrame(out_json.get("interactions", []))
display(df)

,species1_common,species1_scientific,genus_species_combo_sp1,species2_common,species2_scientific,genus_species_combo_sp2,label,effect_on_sp1,effect_on_sp2
0,American Goshawk,Astur atricapillus,Astur atricapillus,Northern Flicker,Colaptes auratus,Colaptes auratus,predation,-1,1
1,American Goshawk,Astur atricapillus,Astur atricapillus,Blue Jay,Cyanocitta cristata,Cyanocitta cristata,predation,-1,1
2,American Goshawk,Astur atricapillus,Astur atricapillus,Steller's Jay,Cyanocorax stelleri,Cyanocorax stelleri,predation,-1,1
3,American Goshawk,Astur atricapillus,Astur atricapillus,Northern Flicker,Colaptes auratus,Colaptes auratus,predation,-1,1
4,American Goshawk,Astur atricapillus,Astur atricapillus,European Starling,Sturnus vulgaris,Sturnus vulgaris,predation,-1,1
5,American Goshawk,Astur atricapillus,Astur atricapillus,American Robin,Turdus migratorius,Turdus migratorius,predation,-1,1
6,American Goshawk,Astur atricapillus,Astur atricapillus,Blue Jay,Cyanocitta cristata,Cyanocitta cristata,predation,-1,1
7,American Goshawk,Astur atricapillus,Astur atricapillus,Gray Jay,Perisoreus canadensis,Perisoreus canadensis,predation,-1,1
8,American Goshawk,Astur atricapillus,Astur atricapillus,Black-backed Woodpecker,Picoides arcticus,Picoides arcticus,predation,-1,1
9,American Goshawk,Astur atricapillus,Astur atricapillus,Northern Flicker,Colaptes auratus,Colaptes auratus,commensalism-scavenge,-1,0


In [39]:
import json

# Load example from file
with open("./data/example_output_minimal.json", "r") as f:
    example_output_minimal = json.load(f)

with open("./data/system_notaxa.txt", "r") as f:
    system_prompt_minimal = f.read().strip()

In [40]:
example_output_minimal

{'interactions': [{'species_1': "Cooper's Hawk",
   'species_2': 'Northern Flicker (Colaptes auratus)',
   'label(type_of_interaction)': 'predation',
   'effect_on_sp1': 1,
   'effect_on_sp2': -1},
  {'species_1': "Cooper's Hawk",
   'species_2': 'Doves (Streptopelia decaocto)',
   'label(type_of_interaction)': 'predation',
   'effect_on_sp1': 1,
   'effect_on_sp2': -1}]}

In [42]:
import pandas as pd
df = pd.DataFrame(example_output_minimal.get("interactions", []))
display(df)

,species_1,species_2,label(type_of_interaction),effect_on_sp1,effect_on_sp2
0,Cooper's Hawk,Northern Flicker (Colaptes auratus),predation,1,-1
1,Cooper's Hawk,Doves (Streptopelia decaocto),predation,1,-1


In [ ]:
# json.dumps(example_output_minimal)


'{"interactions": [{"species_1": "Cooper\'s Hawk", "species_2": "Northern Flicker (Colaptes auratus)", "label(type_of_interaction)": "predation", "effect_on_sp1": 1, "effect_on_sp2": -1}, {"species_1": "Cooper\'s Hawk", "species_2": "Doves (Streptopelia decaocto)", "label(type_of_interaction)": "predation", "effect_on_sp1": 1, "effect_on_sp2": -1}]}'

In [47]:
def call_llm(user_prompt, system_prompt = None, example_pair = None):
    """
    Send prompt to llm and return response
    """

    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    if example_pair:
        example_input, example_output = example_pair
        messages.append({"role": "user", "content": example_input})
        messages.append({"role": "assistant", "content": json.dumps(example_output)})

    messages.append({"role": "user", "content": user_prompt})

    # Define the structured output schema for species interactions
    output_format = {
        "type": "object",
        "properties": {
            "interactions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "species_1": {"type": "string"},
                        "species_2": {"type": "string"},
                        "label(type_of_interaction)": {"type": "string"},
                        "effect_on_sp1": {"type": "integer"},
                        "effect_on_sp2": {"type": "integer"}
                    },
                    "required": [
                        "species_1",
                        "species_2",
                        "label(type_of_interaction)",
                        "effect_on_sp1",
                        "effect_on_sp2"
                    ]
                }
            }
        },
        "required": ["interactions"]
    }
    
    try:
        response = requests.post(
            f"http://{OLLAMA_HOST}/api/chat",
            json={
                "model": LLM_MODEL,
                "stream": False,
                "options": {
                    "num_ctx": 32768
                },
                "messages": messages,
                "format": output_format  # add structured output
            }
        )
        data = response.json()
        return data # response.json()['message']['content']
        
    except Exception as e:
        print(f"Error during llm call: {e}")
        return None

In [48]:
response = call_llm(example_input, system_prompt=system_prompt_minimal, example_pair=(example_input, example_output_minimal))

In [49]:
out = response['message']['content']
out_json = json.loads(out)
print(out_json)

{'interactions': [{'species_1': "Cooper's Hawk", 'species_2': 'House Sparrow', 'label(type_of_interaction)': 'predation', 'effect_on_sp1': 1, 'effect_on_sp2': -1}, {'species_1': "Cooper's Hawk", 'species_2': 'Northern Flicker (Colaptes auratus)', 'label(type_of_interaction)': 'predation', 'effect_on_sp1': 1, 'effect_on_sp2': -1}, {'species_1': "Cooper's Hawk", 'species_2': 'Doves (Streptopelia decaocto)', 'label(type_of_interaction)': 'predation', 'effect_on_sp1': 1, 'effect_on_sp2': -1}, {'species_1': "Cooper's Hawk", 'species_2': 'American Robin', 'label(type_of_interaction)': 'predation', 'effect_on_sp1': 1, 'effect_on_sp2': -1}, {'species_1': "Cooper's Hawk", 'species_2': 'European Starling', 'label(type_of_interaction)': 'predation', 'effect_on_sp1': 1, 'effect_on_sp2': -1}, {'species_1': "Cooper's Hawk", 'species_2': 'Blue Jay (Cyanocitta cristata)', 'label(type_of_interaction)': 'predation', 'effect_on_sp1': 1, 'effect_on_sp2': -1}, {'species_1': "Cooper's Hawk", 'species_2': 'M

In [50]:
df = pd.DataFrame(out_json.get("interactions", []))
display(df)

,species_1,species_2,label(type_of_interaction),effect_on_sp1,effect_on_sp2
0,Cooper's Hawk,House Sparrow,predation,1,-1
1,Cooper's Hawk,Northern Flicker (Colaptes auratus),predation,1,-1
2,Cooper's Hawk,Doves (Streptopelia decaocto),predation,1,-1
3,Cooper's Hawk,American Robin,predation,1,-1
4,Cooper's Hawk,European Starling,predation,1,-1
5,Cooper's Hawk,Blue Jay (Cyanocitta cristata),predation,1,-1
6,Cooper's Hawk,Mourning Dove,predation,1,-1
7,Cooper's Hawk,Northern Mockingbird (Mimus polyglottos),predation,1,-1
